## PostgreSQL DataBase:   **optph_db** 
### --- for FeCl₃ + Additives: High-Throughput Optimization Project
*Sophie Hongfei Liu, 2026-02*


#### 1. connect to the database: optph_db

In [1]:
%%bash
# stop previous server cleanly:
pg_ctl -D /Users/liuh339/Documents/projects/DataBase/pDB stop

# start the database server
pg_ctl -D /Users/liuh339/Documents/projects/DataBase/pDB -l logfile start
# check the database status and restart if needed
pg_ctl -D /Users/liuh339/Documents/projects/DataBase/pDB status

# connect to the optph_db
psql -h localhost -p 5432 -U "liuh339" -d optph_db
\dt


pg_ctl: PID file "/Users/liuh339/Documents/projects/DataBase/pDB/postmaster.pid" does not exist
Is server running?


waiting for server to start.... done
server started
pg_ctl: server is running (PID: 74538)
/opt/homebrew/Cellar/postgresql@17/17.7_1/bin/postgres "-D" "/Users/liuh339/Documents/projects/DataBase/pDB"
            List of relations
 Schema |     Name     | Type  |  Owner  
--------+--------------+-------+---------
 public | additive     | table | liuh339
 public | result_run1  | table | liuh339
 public | run_info     | table | liuh339
 public | summary_run1 | table | liuh339
 public | water_grade  | table | liuh339
(5 rows)



In [81]:
## setup the enviroment
import pandas as pd
from sqlalchemy import create_engine
import psycopg2
from psycopg2.extras import execute_values
import json


%load_ext sql

# create a cursor to Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="optph_db",
    user="liuh339",
)
cur = conn.cursor()

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


#### 2. Add additive table for the libraries

In [13]:
# Drop additive table if exsits
sql_drop = """
DROP TABLE IF EXISTS additive CASCADE;
"""

cur.execute(sql_drop)  # run the sql
conn.commit()     # save the change

# upload the additive table
csv_path = "/Users/liuh339/Library/CloudStorage/OneDrive-PNNL/Documents/projects/Yangang/optph_db_files/additive_library.csv"
table_name = "additive"

df = pd.read_csv(csv_path)
engine = create_engine("postgresql+psycopg2://liuh339:@localhost:5432/optph_db")
df.to_sql(table_name, engine, if_exists="fail", index=False)

# Alter the table to add keys:
sql_alter = """
ALTER TABLE additive
ADD PRIMARY KEY (id);
"""

cur.execute(sql_alter)  # run the sql
conn.commit()     # save the change

38

#### 3. Add water_grade table for the libraries

In [26]:
# Drop additive table if exsits
sql_drop = """
DROP TABLE IF EXISTS water_grade CASCADE;
"""

cur.execute(sql_drop)  # run the sql
conn.commit()     # save the change


# Upload the water_grade table
csv_path = "/Users/liuh339/Library/CloudStorage/OneDrive-PNNL/Documents/projects/Yangang/optph_db_files/water_grade_library.csv"
table_name = "water_grade"

df = pd.read_csv(csv_path)
engine = create_engine("postgresql+psycopg2://liuh339:@localhost:5432/optph_db")
df.to_sql(table_name, engine, if_exists="fail", index=False)

# Alter the table to add keys:
sql_alter = """
ALTER TABLE water_grade
ADD PRIMARY KEY (id);
"""

cur.execute(sql_alter)  # run the sql
conn.commit()     # save the change

4

#### 4. SetUp the run_info table as the metadata

In [50]:
%%bash
psql -h localhost -p 5432 -U "liuh339" -d optph_db

DROP TABLE IF EXISTS run_info CASCADE;

CREATE TABLE run_info(
run_id               bigint PRIMARY KEY,
 design_id           bigint,
 temp_c               NUMERIC,
 stir_rate_rpm        NUMERIC,
 wait_time_min        NUMERIC,
 sample_v_ml          NUMERIC,
 vial_size_ml         NUMERIC,
 KOH_conc_m           NUMERIC,
 FeCl3_conc_m         NUMERIC,
 KOH_addn_type        text,
 KOH_addn_method      text,
 KOH_lookahead        boolean,
 FeCl3_addn_type      text,
 FeCl3_addn_method    text,
 FeCl3_lookahead      boolean,
 additive_addn_type   text,
 additive_addn_method text,
 water_addn_method    text,
 additive_info        JSONB,
 raw_home_folder      text,
 raw_run_folder       text,
 result_table         text,
 summary_table        text,
 run_date             date,
 measurement_date     date,
 operator             text,
 image_name_format    text
);

\copy run_info FROM '/Users/liuh339/Library/CloudStorage/OneDrive-PNNL/Documents/projects/Yangang/optph_db_files/run_info.csv' WITH (FORMAT csv, HEADER true);


DROP TABLE
CREATE TABLE
COPY 3


#### 5. Add result_run and summary_run tables for each run

In [49]:
# Add result_run
table_name = "result_run1"
csv_path = "/Users/liuh339/Library/CloudStorage/OneDrive-PNNL/Documents/projects/Yangang/optph_db_files/run1/result_run1.csv"


# Drop table if exists
sql_drop = f"""
DROP TABLE IF EXISTS {table_name} CASCADE;
"""
cur.execute(sql_drop)
conn.commit()

# Upload the table from CSV
df = pd.read_csv(csv_path)

engine = create_engine("postgresql+psycopg2://liuh339:@localhost:5432/optph_db")
df.to_sql(table_name, engine, if_exists="fail", index=False)

# Alter the table to add primary key
sql_alter = f"""
ALTER TABLE {table_name}
ADD PRIMARY KEY (id),
ADD FOREIGN KEY (water_grade_id) REFERENCES water_grade (id),
ADD FOREIGN KEY (additive_id) REFERENCES additive (id),
ADD FOREIGN KEY (run_id) REFERENCES run_info (run_id);
"""

cur.execute(sql_alter)
conn.commit()

48

In [69]:
# Add summary_run
table_name = "summary_run1"
csv_path = "/Users/liuh339/Library/CloudStorage/OneDrive-PNNL/Documents/projects/Yangang/optph_db_files/run1/summary_run1.csv"


# Drop table if exists
sql_drop = f"""
DROP TABLE IF EXISTS {table_name} CASCADE;
"""
cur.execute(sql_drop)
conn.commit()

# Upload the  table from CSV
df = pd.read_csv(csv_path)

engine = create_engine("postgresql+psycopg2://liuh339:@localhost:5432/optph_db")
df.to_sql(table_name, engine, if_exists="fail", index=False)

# Alter the table to add primary key
sql_alter = f"""
ALTER TABLE {table_name}
  ALTER COLUMN run_id TYPE integer,
  ADD PRIMARY KEY (run_id),
  ADD FOREIGN KEY (run_id) REFERENCES run_info (run_id);
"""

cur.execute(sql_alter)
conn.commit()

#### 5. Add each run's information to run_info table

In [33]:
# Add the information from Record_Form
# read in table to dict format
df = pd.read_csv(
    "/Users/liuh339/Library/CloudStorage/OneDrive-PNNL/Documents/projects/Yangang/optph_db_files/Record_Form_experiment_run1.csv",
    usecols=[0, 2]    # keep columns 0 and 2 (1st and 3rd)

)

row_dict = df.set_index("records")["value"].to_dict()

row_dict


# add to run_info table in the database
cols = list(row_dict.keys())
values = [row_dict[c] for c in cols]

sql = f"""
    INSERT INTO run_info ({', '.join(cols)})
    VALUES ({', '.join(['%s'] * len(cols))})
"""

cur.execute(sql, values)
conn.commit()

In [82]:
# add the additives information
csv_path = "/Users/liuh339/Library/CloudStorage/OneDrive-PNNL/Documents/projects/Yangang/optph_db_files/run1/result_run1.csv"
run_id = 1  # identifies the row in run_info

df = pd.read_csv(csv_path, usecols=["additive_id", "add_final_concentration_M"])
df = df[df["additive_id"] != 0]
df = df.drop_duplicates()

records = [
    {
        "additive_id": int(row["additive_id"]),
        "concentration": float(row["add_final_concentration_M"]),
    }
    for _, row in df.iterrows()
]


sql = """
UPDATE run_info
SET additive_info = %s
WHERE run_id = %s;
"""

cur.execute(sql, [json.dumps(records), run_id])
conn.commit()

cur.close()
conn.close()

In [85]:
%%bash
psql -h localhost -p 5432 -U "liuh339" -d optph_db

\dt

select * from run_info;

DROP TABLE IF EXISTS "result_run1.csv" CASCADE;
DROP TABLE IF EXISTS experiment_info CASCADE;

\dt


             List of relations
 Schema |      Name       | Type  |  Owner  
--------+-----------------+-------+---------
 public | additive        | table | liuh339
 public | experiment_info | table | liuh339
 public | result_run1     | table | liuh339
 public | result_run1.csv | table | liuh339
 public | run_info        | table | liuh339
 public | summary_run1    | table | liuh339
 public | water_grade     | table | liuh339
(7 rows)

 run_id | design_id | temp_c | stir_rate_rpm | wait_time_min | sample_v_ml | vial_size_ml | koh_conc_m | fecl3_conc_m | koh_addn_type | koh_addn_method | koh_lookahead | fecl3_addn_type | fecl3_addn_method | fecl3_lookahead | additive_addn_type | additive_addn_method |   water_addn_method   |                                     additive_info                                      | raw_home_folder | raw_run_folder | result_table | summary_table |  run_date  | measurement_date |    operator    |          image_name_format           
--------+-----------+----

In [86]:
cur.close()
conn.close()